In [1]:
!pip install transformers datasets accelerate evaluate scikit-learn --quiet


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
import evaluate
import json
import os

c:\Users\hp\Desktop\Research-System\training\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Load dataset
df = pd.read_csv(r"C:\Users\hp\Desktop\final_dataset.csv")

# Get sorted lists of unique categories (sorted = deterministic order)
main_categories = sorted(df['main_label'].unique())
sub_categories = sorted(df['sub_label'].unique())


print(f"Main categories ({len(main_categories)}): {main_categories}")
print(f"Sub categories ({len(sub_categories)}): {sub_categories}")
print()

Main categories (7): ['Biology & Medicine', 'Chemistry', 'Computer Science', 'Economics & Business', 'Engineering', 'Mathematics', 'Physics']
Sub categories (42): ['Accounting', 'Algebra', 'Analysis', 'Analytical Chemistry', 'Artificial Intelligence', 'Astrophysics', 'Cardiology', 'Chemical Engineering', 'Civil & Structural Engineering', 'Combinatorics', 'Condensed Matter', 'Cryptography & Security', 'Data Structures & Algorithms', 'Econometrics', 'Electrical & Electronic Engineering', 'Electrochemistry', 'Finance', 'Genetics & Molecular Biology', 'High Energy Physics', 'Immunology & Microbiology', 'Industrial & Manufacturing Engineering', 'Inorganic Chemistry', 'Machine Learning', 'Management', 'Management Information Systems', 'Marketing', 'Mathematical Physics', 'Mechanical Engineering', 'Natural Language Processing', 'Neuroscience', 'Numerical Analysis', 'Oncology', 'Optimization & Control', 'Organic Chemistry', 'Pharmacology', 'Physical and Theoretical Chemistry', 'Probability', '

In [7]:
# Create mapping dictionaries: text → integer
main_label2id = {label: idx for idx, label in enumerate(main_categories)}
id2main_label = {idx: label for label, idx in main_label2id.items()}

sub_label2id  = {label: idx for idx, label in enumerate(sub_categories)}
id2sub_label  = {idx: label for label, idx in sub_label2id.items()}

# Add encoded columns to the DataFrame
df['main_label_id'] = df['main_label'].map(main_label2id)
df['sub_label_id']  = df['sub_label'].map(sub_label2id)

# Quick sanity check: a few rows before and after
print("Sample encoding:")
sample = df[['main_label', 'main_label_id', 'sub_label', 'sub_label_id']].head(5)
sample

Sample encoding:


,main_label,main_label_id,sub_label,sub_label_id
0,Mathematics,5,Analysis,2
1,Engineering,4,Chemical Engineering,7
2,Mathematics,5,Analysis,2
3,Physics,6,Condensed Matter,10
4,Chemistry,1,Spectroscopy,40


In [ ]:
# 80/10/10 Stratified Train / Validation / Test Split

# First split: 80% train, 20% temp (val + test combined)
train_df, temp_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df['sub_label'],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    stratify=temp_df['sub_label'],
    random_state=42
)

# Report sizes
print(f"Full dataset: {len(df)} papers")
print(f"Training:   {len(train_df)} papers ({len(train_df)/len(df)*100:.1f}%)")
print(f"Validation: {len(val_df)} papers ({len(val_df)/len(df)*100:.1f}%)")
print(f"Test:       {len(test_df)} papers ({len(test_df)/len(df)*100:.1f}%)")
print()
print("Training set main category distribution:")
print(train_df['main_label'].value_counts())

In [ ]:
# Tokenization: convert text to numbers SciBERT understands

# Load the SciBERT tokenizer (downloads vocabulary the first time)
tokenizer = AutoTokenizer.from_pretrained("allenai/scibert_scivocab_uncased")

# L1 tokenization function: abstract only
def tokenize_l1(batch):
    return tokenizer(
        batch["abstract"],
        truncation=True,          # cut text that exceeds 512 tokens
        padding="max_length",     # pad shorter texts to exactly 512 tokens
        max_length=512
    )

# L2 tokenization function: hint + abstract
def tokenize_l2(batch):
    # Build hint strings for every row in the batch
    hints = [
        f"Main category: {main}\n\n{abstract}"
        for main, abstract in zip(batch["main_label"], batch["abstract"])
    ]
    return tokenizer(
        hints,
        truncation=True,
        padding="max_length",
        max_length=512
    )

# Convert pandas DataFrames to a HuggingFace Datasets
train_dataset = Dataset.from_pandas(train_df)
val_dataset   = Dataset.from_pandas(val_df)

# Tokenize L1 datasets (abstract only)
tokenized_train_l1 = train_dataset.map(tokenize_l1, batched=True)
tokenized_val_l1   = val_dataset.map(tokenize_l1, batched=True)

# Tokenize L2 datasets (hint + abstract)
tokenized_train_l2 = train_dataset.map(tokenize_l2, batched=True)
tokenized_val_l2   = val_dataset.map(tokenize_l2, batched=True)

# Rename label columns to "labels" (Trainer requirement)
tokenized_train_l1 = tokenized_train_l1.rename_column("main_label_id", "labels")
tokenized_val_l1   = tokenized_val_l1.rename_column("main_label_id", "labels")
tokenized_train_l2 = tokenized_train_l2.rename_column("sub_label_id", "labels")
tokenized_val_l2   = tokenized_val_l2.rename_column("sub_label_id", "labels")

# Set format to PyTorch tensors
columns = ["input_ids", "attention_mask", "labels"]
for ds in [tokenized_train_l1, tokenized_val_l1, tokenized_train_l2, tokenized_val_l2]:
    ds.set_format("torch", columns=columns)

print("Tokenization complete.")
print(f"Train L1: {len(tokenized_train_l1)} examples")
print(f"Val   L1: {len(tokenized_val_l1)} examples")
print(f"Train L2: {len(tokenized_train_l2)} examples")
print(f"Val   L2: {len(tokenized_val_l2)} examples")

In [ ]:
# Training SciBERT L1 (Main Category)

# Define evaluation metrics
def compute_metrics(eval_pred):
    predictions, labels = eval_pred                          # predictions: logits; labels: ints
    predictions = predictions.argmax(axis=-1)                # pick the highest-scoring class
    acc  = accuracy_score(labels, predictions)
    f1   = f1_score(labels, predictions, average="weighted")
    return {"accuracy": acc, "f1": f1}

# Load pre‑trained SciBERT with a 7‑class head
l1_model = AutoModelForSequenceClassification.from_pretrained(
    "allenai/scibert_scivocab_uncased",
    num_labels=len(main_label2id),            # 7 main categories
    problem_type="single_label_classification"
)

# Training arguments (optimized for Kaggle T4 GPU)
training_args_l1 = TrainingArguments(
    output_dir="./results_l1",                # where checkpoints and logs go

    # Training loop settings
    num_train_epochs=3,                       # go through the whole dataset 3 times
    learning_rate=2e-5,                       # typical fine‑tuning rate for BERT models
    weight_decay=0.01,                        # L2 regularisation (helps prevent overfitting)

    # Batch sizes, chosen to fit in 16 GB GPU memory
    per_device_train_batch_size=16,           # 16 papers per GPU step
    per_device_eval_batch_size=32,            # larger batch for evaluation (no gradients)

    # Evaluation & checkpointing
    evaluation_strategy="epoch",              # evaluate after every epoch
    save_strategy="epoch",                    # save a checkpoint after every epoch
    load_best_model_at_end=True,              # after training, restore the best checkpoint
    metric_for_best_model="f1",               # use weighted F1 to decide what is "best"

    # Logging
    logging_strategy="steps",
    logging_steps=50,                         # log loss every 50 training steps

    # Performance
    fp16=True,                                # mixed precision, roughly doubles speed on T4
    report_to="none"                          # disable wandb / other external loggers
)

# Create the Trainer
l1_trainer = Trainer(
    model=l1_model,
    args=training_args_l1,
    train_dataset=tokenized_train_l1,
    eval_dataset=tokenized_val_l1,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

# Train!
l1_trainer.train()

# Final evaluation on the validation set
l1_results = l1_trainer.evaluate()
print("\nL1 Validation Results:")
for key, val in l1_results.items():
    print(f"  {key}: {val:.4f}")

In [ ]:
# Training SciBERT L2 (Subcategory with Hint)

# Load pre‑trained SciBERT with a 42‑class head
l2_model = AutoModelForSequenceClassification.from_pretrained(
    "allenai/scibert_scivocovab_uncased",     # same vocabulary as L1
    num_labels=len(sub_label2id),            # 42 subcategories
    problem_type="single_label_classification"
)

# Training arguments (same hyperparameters as L1)
training_args_l2 = TrainingArguments(
    output_dir="./results_l2",                # separate folder for L2 checkpoints

    # Training loop settings
    num_train_epochs=3,                       # 3 full passes over the training data
    learning_rate=2e-5,                       # standard fine‑tuning rate for BERT
    weight_decay=0.01,                        # L2 regularisation

    # Batch sizes – chosen to fit in 16 GB GPU memory
    per_device_train_batch_size=16,           # 16 papers per GPU step
    per_device_eval_batch_size=32,            # larger batch for evaluation (no gradients)

    # Evaluation & checkpointing
    evaluation_strategy="epoch",              # evaluate after every epoch
    save_strategy="epoch",                    # save a checkpoint after every epoch
    load_best_model_at_end=True,              # restore the best checkpoint after training
    metric_for_best_model="f1",              # use weighted F1 to pick the best checkpoint

    # Logging
    logging_strategy="steps",
    logging_steps=50,                         # log training loss every 50 steps

    # Performance
    fp16=True,                                # mixed precision – faster training on T4
    report_to="none"                          # disable external logging
)

# Create the L2 Trainer
l2_trainer = Trainer(
    model=l2_model,
    args=training_args_l2,
    train_dataset=tokenized_train_l2,        # hint + abstract, labels = sub_label_id
    eval_dataset=tokenized_val_l2,           # same structure for validation
    tokenizer=tokenizer,                     # same tokenizer used for both L1 and L2
    compute_metrics=compute_metrics          # defined earlier (accuracy + weighted F1)
)

# Train
l2_trainer.train()

# Final evaluation on the validation set
l2_results = l2_trainer.evaluate()
print("\nL2 Validation Results:")
for key, val in l2_results.items():
    print(f"  {key}: {val:.4f}")

In [ ]:
# Save Models, Tokenizer, and Label Mappings

# Save L1 model & tokenizer
save_path_l1 = "./scibert_l1"
l1_trainer.save_model(save_path_l1)          # saves pytorch_model.bin + config.json
tokenizer.save_pretrained(save_path_l1)      # saves vocab.txt + tokenizer.json

# L1 label mapping (7 main categories)
with open(f"{save_path_l1}/label_mapping.json", "w") as f:
    json.dump({"id2label": id2main_label, "label2id": main_label2id}, f, indent=2)

print(f"L1 model saved to {save_path_l1}/")

# Save L2 model & tokenizer
save_path_l2 = "./scibert_l2"
l2_trainer.save_model(save_path_l2)
tokenizer.save_pretrained(save_path_l2)

# L2 label mapping (42 subcategories)
with open(f"{save_path_l2}/label_mapping.json", "w") as f:
    json.dump({"id2label": id2sub_label, "label2id": sub_label2id}, f, indent=2)

print(f"L2 model saved to {save_path_l2}/")

# Also save the test set for final evaluation later
test_df.to_csv("test_set.csv", index=False)
print(f"Test set saved ({len(test_df)} papers)")

2. Inference Threshold (L2 Confidence for Graceful Degradation)
This is the more project‑specific tuning – and we do plan to tune it.

What it is
After L2 predicts a subcategory, we have a confidence score (the softmax probability of the winning subcategory). The current threshold is 0.6 (60%). If the confidence is below 0.6, we hide the subcategory and show “Unclassified”.

Why tune it?
The optimal threshold depends on the model’s real‑world behaviour:

Too low → you show subcategories when the model is actually unsure (bad user trust).

Too high → you hide subcategories too often, even when the model is confidently correct (frustrating for users).

How we’ll tune it (after training)
We’ll use the validation set to plot precision/recall at different thresholds and pick the one that balances false‑positives and false‑negatives according to your project’s goal (more conservative vs. more informative). This is a small, fast computation – no GPU needed.

Where it fits in the notebook
We’ll add a separate cell after both models are trained and saved. That cell will:

Run L2 inference on the validation set.

Collect the winning subcategory confidence for each paper.

Compare against the true sub‑label to see whether the model was correct.

Scan thresholds from 0.0 to 1.0 and compute metrics (accuracy, % unclassified, etc.).

Let you choose the final threshold – which then goes into your classifier.py.

We’ll write that cell together once we’ve covered the whole training pipeline.

The training notebook is complete up through saving the two SciBERT models, the tokenizer, and the label mappings. The final steps – and I want to make sure you’re aware of them – are:

Threshold‑tuning cell (using the validation set to pick the optimal L2‑confidence cut‑off)

Final evaluation cell (running on the held‑out test set for an honest accuracy/F1 number)

A closing markdown cell with download instructions (what to grab from Kaggle and where to put it locally)